# 03 — Metrics

EER (Equal Error Rate), DET curves, and per-attack breakdown utilities.
These functions are imported verbatim into notebooks 05 and 06.

## Imports

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
from sklearn.metrics import roc_curve

print("Imports OK")

## EER

In [ ]:
def compute_eer(labels, scores):
    """
    labels : array-like of int  (0=bonafide, 1=spoof)
    scores : array-like of float (higher => more likely spoof)
    Returns (eer: float, threshold: float)
    """
    fpr, tpr, thresholds = roc_curve(labels, scores, pos_label=1)
    fnr = 1.0 - tpr
    idx = np.nanargmin(np.abs(fnr - fpr))
    eer = (fpr[idx] + fnr[idx]) / 2.0
    return float(eer), float(thresholds[idx])

print("compute_eer defined.")

## DET Curve

In [ ]:
def plot_det(labels, scores, title="DET Curve", ax=None, color="steelblue", label=None):
    """Plots a Detection Error Tradeoff curve on normal-deviate axes."""
    fpr, tpr, _ = roc_curve(labels, scores, pos_label=1)
    fnr  = 1.0 - tpr
    eps  = 1e-6
    x_nd = norm.ppf(np.clip(fpr, eps, 1 - eps))
    y_nd = norm.ppf(np.clip(fnr, eps, 1 - eps))

    if ax is None:
        _, ax = plt.subplots(figsize=(6, 6))

    ticks_pct = [0.1, 0.2, 0.5, 1, 2, 5, 10, 20, 40]
    tv = norm.ppf([t / 100 for t in ticks_pct])

    # EER diagonal (where FPR == FNR)
    ax.plot([tv[0], tv[-1]], [tv[0], tv[-1]],
            color="gray", linestyle="--", linewidth=0.8, alpha=0.6)

    ax.plot(x_nd, y_nd, color=color, linewidth=1.5, label=label)

    ax.set_xlim(tv[0], tv[-1])
    ax.set_ylim(tv[0], tv[-1])
    ax.set_xticks(tv); ax.set_xticklabels([str(t) for t in ticks_pct], rotation=45, ha="right")
    ax.set_yticks(tv); ax.set_yticklabels([str(t) for t in ticks_pct])
    ax.set_xlabel("False Alarm Rate (%)")
    ax.set_ylabel("Miss Rate (%)")
    ax.set_title(title)
    ax.grid(True, linestyle="--", alpha=0.4)
    if label:
        ax.legend()
    return ax

print("plot_det defined.")

## Per-Attack EER

In [ ]:
def per_attack_eer(labels, scores, attack_ids):
    """
    For each spoof attack, compute EER against ALL bonafide samples.
    Bonafide samples are identified by label == 0.
    Returns {attack_id: eer_float} sorted ascending by EER.
    """
    labels     = np.asarray(labels)
    scores     = np.asarray(scores)
    attack_ids = np.asarray(attack_ids)

    bona_mask = labels == 0          # all bonafide samples

    results = {}
    for atk in np.unique(attack_ids):
        spoof_mask = (attack_ids == atk) & (labels == 1)
        if spoof_mask.sum() == 0:    # skip groups with no spoof samples
            continue
        mask = bona_mask | spoof_mask
        eer, _ = compute_eer(labels[mask], scores[mask])
        results[atk] = eer

    return dict(sorted(results.items(), key=lambda x: x[1]))

print("per_attack_eer defined.")

## Sanity Check

In [ ]:
# Synthetic data: two Gaussian blobs
np.random.seed(42)
n = 2000
syn_labels = np.array([0] * n + [1] * n)
syn_scores = np.concatenate([
    np.random.normal(0.3, 0.15, n),
    np.random.normal(0.7, 0.15, n),
])
syn_scores = np.clip(syn_scores, 0, 1)

eer, thr = compute_eer(syn_labels, syn_scores)
print(f"Synthetic EER : {eer * 100:.2f}%   threshold = {thr:.4f}")

fig, ax = plt.subplots(figsize=(5, 5))
plot_det(syn_labels, syn_scores, title="Synthetic DET", ax=ax, label=f"EER = {eer*100:.2f}%")
plt.tight_layout()
plt.show()